In [1]:
import pandas as pd

In [2]:
df = pd.DataFrame({
    "COVID": ["Yes", "No", "Yes", "No", "Yes", "No", "Yes", "Yes", "No", "No"],
    "Flu": ["No", "Yes", "Yes", "No", "No", "No", "No", "No","Yes", "Yes"],
    "Fever": ["Yes","Yes","Yes","No", "Yes", "Yes", "Yes", "No", "Yes","No" ]
})
for cols in df.columns:
    df[cols] = (df[cols] == "Yes").astype(int)

print(df)

   COVID  Flu  Fever
0      1    0      1
1      0    1      1
2      1    1      1
3      0    0      0
4      1    0      1
5      0    0      1
6      1    0      1
7      1    0      0
8      0    1      1
9      0    1      0


In [3]:
#features and label
X = df[["COVID", "Flu"]] 
y = df["Fever"] 

In [4]:
#prior probability
fever_yes = y.mean()          
fever_no  = 1 - fever_yes     
print(f"Fever: P(Yes)={fever_yes:.2f}  P(No)={fever_no:.2f}")


Fever: P(Yes)=0.70  P(No)=0.30


In [5]:
#conditional probability
features = ["COVID", "Flu"]

cond_probs = {}

for feat in features:
    p_feat_given_yes = df.loc[y == 1, feat].mean()  # 4/7
    p_feat_given_no  = df.loc[y == 0, feat].mean()  # 2/3
   
    cond_probs[feat] = {1: p_feat_given_yes, 0: p_feat_given_no}  #dict to store these values
    print(f"{feat}:  P({feat}=Yes | Fever=Yes) = {p_feat_given_yes:.4f}  "
          f"P({feat}=Yes | Fever=No) = {p_feat_given_no:.4f}")

COVID:  P(COVID=Yes | Fever=Yes) = 0.5714  P(COVID=Yes | Fever=No) = 0.3333
Flu:  P(Flu=Yes | Fever=Yes) = 0.4286  P(Flu=Yes | Fever=No) = 0.3333


In [6]:
def naive_bayes_predict(sample: dict) -> str:
    score_yes = fever_yes
    score_no  = fever_no

    for feat, val in sample.items():
        p_yes = cond_probs[feat][1] if val == 1 else (1 - cond_probs[feat][1])
        p_no  = cond_probs[feat][0] if val == 1 else (1 - cond_probs[feat][0])
        score_yes *= p_yes
        score_no  *= p_no

    return "Yes" if score_yes > score_no else "No"

#Evaluate on entire dataset
y_true = []
y_pred = []

print("Row  COVID  Flu  Actual  Predicted  Correct?")
print("-" * 47)

for i, row in X.iterrows():
    sample    = {"COVID": row["COVID"], "Flu": row["Flu"]}
    actual    = "Yes" if y[i] == 1 else "No"
    predicted = naive_bayes_predict(sample)
    correct   = "Correct" if actual == predicted else "No"

    y_true.append(actual)
    y_pred.append(predicted)

    print(f"  {i+1:<4} {row['COVID']:<6} {row['Flu']:<4} {actual:<7} {predicted:<10} {correct}")

#accuracy
correct_count = sum(t == p for t, p in zip(y_true, y_pred))
accuracy      = correct_count / len(y_true)
print(f"\nAccuracy: {correct_count}/{len(y_true)} = {accuracy:.2%}")

#Confusion Matrix
TP = sum(t == "Yes" and p == "Yes" for t, p in zip(y_true, y_pred))
TN = sum(t == "No"  and p == "No"  for t, p in zip(y_true, y_pred))
FP = sum(t == "No"  and p == "Yes" for t, p in zip(y_true, y_pred))
FN = sum(t == "Yes" and p == "No"  for t, p in zip(y_true, y_pred))

print(f"""
Confusion Matrix:
                 Predicted Yes   Predicted No
Actual Yes           {TP} (TP)         {FN} (FN)
Actual No            {FP} (FP)         {TN} (TN)
""")

#Precision, Recall, F1
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"Precision : {TP}/({TP}+{FP}) = {precision:.2%}")
print(f"Recall    : {TP}/({TP}+{FN}) = {recall:.2%}")
print(f"F1 Score  : {f1:.2%}")


Row  COVID  Flu  Actual  Predicted  Correct?
-----------------------------------------------
  1    1      0    Yes     Yes        Correct
  2    0      1    Yes     Yes        Correct
  3    1      1    Yes     Yes        Correct
  4    0      0    No      Yes        No
  5    1      0    Yes     Yes        Correct
  6    0      0    Yes     Yes        Correct
  7    1      0    Yes     Yes        Correct
  8    1      0    No      Yes        No
  9    0      1    Yes     Yes        Correct
  10   0      1    No      Yes        No

Accuracy: 7/10 = 70.00%

Confusion Matrix:
                 Predicted Yes   Predicted No
Actual Yes           7 (TP)         0 (FN)
Actual No            3 (FP)         0 (TN)

Precision : 7/(7+3) = 70.00%
Recall    : 7/(7+0) = 100.00%
F1 Score  : 82.35%
